# QUASR Finite-Beta Pipeline
Load 3 QA equilibria, visualize vacuum stage, run fixed-boundary VMEC at 2% beta, extract MHD quantities.

In [ ]:
import os
import sys
import warnings

# Ensure tme/ modules are importable (run notebook from tme/ directory)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from load_quasr import load_quasr_equilibrium
from vmec_runner import prepare_vmec_input, run_vmec, run_vmec_to_target_beta
from visualize import (
    plot_vacuum_3d, plot_cross_sections,
    plot_vmec_profiles, plot_flux_surfaces, plot_finite_beta_3d,
)
from analysis import extract_vmec_results, run_boozer_analysis, save_results

# Configuration
QUASR_DIR = "QUASR_eq"
OUTPUT_BASE = "output"
EQUILIBRIA = {
    "0010273": {"symmetry_type": "QA"},
    "0019548": {"symmetry_type": "QA"},
    "0358936": {"symmetry_type": "QA"},
}

## Step 1: Load and Visualize Vacuum Equilibria

In [ ]:
equilibria = {}

for model_id, config in EQUILIBRIA.items():
    print(f"\n{'='*60}")
    print(f"Loading {model_id} ({config['symmetry_type']})")
    print(f"{'='*60}")

    serial_file = os.path.join(QUASR_DIR, f"serial{model_id}.json")
    eq = load_quasr_equilibrium(
        serial_file,
        symmetry_type=config["symmetry_type"],
    )
    equilibria[model_id] = eq

    meta = eq["metadata"]
    print(f"  nfp={meta['nfp']}, symmetry={meta['symmetry_type']}")
    print(f"  Surface: {eq['surface']}")
    print(f"  Coils: {len(eq['coils'])}")

    # 3D vacuum visualization
    fig_3d = plot_vacuum_3d(eq["surface"], eq["coils"], eq["bs"])
    fig_3d.update_layout(title=f"{model_id} — Vacuum 3D")
    fig_3d.show()

    # Cross-sections
    fig_cs = plot_cross_sections(eq["surface"], meta["nfp"])
    fig_cs.suptitle(f"{model_id} — Cross-Sections")
    fig_cs.show()

## Step 2: Run Fixed-Boundary VMEC at 2% Beta

In [ ]:
wout_paths = {}

for model_id, eq in equilibria.items():
    print(f"\n{'='*60}")
    print(f"Running VMEC for {model_id}")
    print(f"{'='*60}")

    output_dir = os.path.join(OUTPUT_BASE, model_id)
    input_template = os.path.join(QUASR_DIR, f"input.{model_id}")

    try:
        wout_path = run_vmec_to_target_beta(
            eq["surface"],
            eq["metadata"],
            output_dir,
            input_template=input_template,
        )
        wout_paths[model_id] = wout_path
        print(f"  VMEC converged! wout: {wout_path}")

    except Exception as e:
        warnings.warn(f"VMEC failed for {model_id}: {e}")
        print(f"  FAILED: {e}")

print(f"\nSuccessful runs: {len(wout_paths)}/{len(equilibria)}")

## Step 3: Extract MHD Quantities and Boozer Analysis

In [ ]:
all_results = {}

for model_id, wout_path in wout_paths.items():
    print(f"\n{'='*60}")
    print(f"Analyzing {model_id}")
    print(f"{'='*60}")

    eq = equilibria[model_id]
    results = extract_vmec_results(wout_path)
    print(f"  beta = {results['betatot']:.4e}")
    print(f"  ier_flag = {results['ier_flag']}")
    print(f"  fsql = {results['fsql']:.2e}")
    print(f"  iota(0) = {results['iotaf'][0]:.4f}, iota(1) = {results['iotaf'][-1]:.4f}")

    # Boozer analysis
    try:
        boozer = run_boozer_analysis(wout_path)
        results.update(boozer)
        print(f"  Boozer: {len(boozer['bmnc_b'])} dominant modes")
        if boozer["epsilon_eff"]:
            print(f"  epsilon_eff(LCFS) = {boozer['epsilon_eff'][-1]:.4e}")
    except Exception as e:
        warnings.warn(f"Boozer failed for {model_id}: {e}")
        print(f"  Boozer FAILED: {e}")

    # Save results
    output_dir = os.path.join(OUTPUT_BASE, model_id)
    save_results(results, eq["metadata"], output_dir, wout_path=wout_path)
    print(f"  Results saved to {output_dir}/results.json")

    all_results[model_id] = results

## Step 4: Finite-Beta Visualization

In [ ]:
for model_id, wout_path in wout_paths.items():
    print(f"\n{'='*60}")
    print(f"Finite-beta plots for {model_id}")
    print(f"{'='*60}")

    # Profile plots (iota, pressure, Mercier, magnetic well)
    fig_prof = plot_vmec_profiles(wout_path)
    fig_prof.suptitle(f"{model_id} — VMEC Profiles", y=1.02)
    fig_prof.show()

    # Flux surface cross-sections
    fig_flux = plot_flux_surfaces(wout_path)
    fig_flux.suptitle(f"{model_id} — Flux Surfaces")
    fig_flux.show()

    # 3D finite-beta surface colored by |B|
    fig_3d = plot_finite_beta_3d(wout_path)
    fig_3d.update_layout(title=f"{model_id} — Finite-Beta |B|")
    fig_3d.show()

## Summary

In [ ]:
import pandas as pd

rows = []
for model_id, results in all_results.items():
    row = {
        "model_id": model_id,
        "symmetry": equilibria[model_id]["metadata"]["symmetry_type"],
        "nfp": results.get("nfp", ""),
        "beta_target": 0.02,
        "beta_achieved": f"{results['betatot']:.4e}",
        "iota_axis": f"{results['iotaf'][0]:.4f}",
        "iota_edge": f"{results['iotaf'][-1]:.4f}",
        "Mercier_min": f"{min(results['DMerc']):.4e}",
        "converged": results.get("ier_flag", -1) == 0,
        "fsql": f"{results['fsql']:.2e}",
    }
    if "epsilon_eff" in results and results["epsilon_eff"]:
        row["eps_eff_LCFS"] = f"{results['epsilon_eff'][-1]:.4e}"
    rows.append(row)

df = pd.DataFrame(rows)
print("Pipeline complete.\n")
df